## Training

In [ ]:
!pip -q install -U "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.0" \
                 "trl>=0.11.0" "peft>=0.11.1" "bitsandbytes>=0.46.1" "safetensors>=0.4.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 23.3 MB/s eta 0:00:00


In [ ]:
import os

# Must be set BEFORE importing torch/transformers/trl/accelerate
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.environ["TRANSFORMERS_NO_BF16"] = "1"   # hard-disable bf16 paths in transformers


import torch
torch.set_default_dtype(torch.float16)

print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

CUDA: 12.8
GPU: Tesla T4


In [ ]:
RUN_COUNTRY = "US"   # choose "MEX" or "US". Later add more countries

if RUN_COUNTRY == "MEX":
    DATA_COUNTRY_CODE = "MEX"
    ADAPTER_TAG = "MEX"
elif RUN_COUNTRY == "US":
    DATA_COUNTRY_CODE = "USA"
    ADAPTER_TAG = "US"
else:
    raise ValueError("RUN_COUNTRY must be either 'MEX' or 'US'")

In [ ]:
#!pip install trl
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model#, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# set output directory here
OUTPUT_DIR = "/content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3"

In [ ]:
#!pip install -U "huggingface_hub[cli]"
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import json
import random
from pathlib import Path

# ============================
# Step 2: Train/eval split
# ============================

# set directory here

DATA_DIR = Path("/content/drive/MyDrive/DPO")

US_RAW_FILE  = DATA_DIR / "D_syn_USA_350.jsonl"
MEX_RAW_FILE = DATA_DIR / "D_syn_MEX_350.jsonl"

US_TRAIN_FILE  = DATA_DIR / "D_syn_USA_train.jsonl"
US_EVAL_FILE   = DATA_DIR / "D_syn_USA_eval.jsonl"

MEX_TRAIN_FILE = DATA_DIR / "D_syn_MEX_train.jsonl"
MEX_EVAL_FILE  = DATA_DIR / "D_syn_MEX_eval.jsonl"


#train-test split here. adjust as needed
TRAIN_FRAC = 0.80
SEED = 42


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def split_country_file(raw_path, train_path, eval_path, country, train_frac=0.80, seed=42):
    rows = load_jsonl(raw_path)

    # Add useful metadata for later validation.
    for i, row in enumerate(rows):
        row.setdefault("country", country)
        row.setdefault("item_id", f"{country}_{i:04d}")

    rng = random.Random(seed)
    rng.shuffle(rows)

    n_train = int(len(rows) * train_frac)

    train_rows = rows[:n_train]
    eval_rows = rows[n_train:]

    write_jsonl(train_rows, train_path)
    write_jsonl(eval_rows, eval_path)

    print(f"{country}: {len(rows)} total")
    print(f"  train: {len(train_rows)} -> {train_path}")
    print(f"  eval:  {len(eval_rows)} -> {eval_path}")

    return train_rows, eval_rows


us_train, us_eval = split_country_file(
    raw_path=US_RAW_FILE,
    train_path=US_TRAIN_FILE,
    eval_path=US_EVAL_FILE,
    country="US",
    train_frac=TRAIN_FRAC,
    seed=SEED,
)

mex_train, mex_eval = split_country_file(
    raw_path=MEX_RAW_FILE,
    train_path=MEX_TRAIN_FILE,
    eval_path=MEX_EVAL_FILE,
    country="Mexico",
    train_frac=TRAIN_FRAC,
    seed=SEED,
)

US: 350 total
  train: 280 -> /content/drive/MyDrive/DPO/D_syn_USA_train.jsonl
  eval:  70 -> /content/drive/MyDrive/DPO/D_syn_USA_eval.jsonl
Mexico: 350 total
  train: 280 -> /content/drive/MyDrive/DPO/D_syn_MEX_train.jsonl
  eval:  70 -> /content/drive/MyDrive/DPO/D_syn_MEX_eval.jsonl


In [ ]:
import json, math, os

IN_FILE  = f"/content/drive/MyDrive/DPO/D_syn_{DATA_COUNTRY_CODE}_train.jsonl"
OUT_FILE = f"/content/drive/MyDrive/DPO/D_syn_{DATA_COUNTRY_CODE}_train_with_ref.jsonl"


# Keep these modest for Colab
MAX_PROMPT_TOKENS = 256
MAX_COMPLETION_TOKENS = 256   # chosen/rejected truncation
BATCH_SIZE = 1                # increase only if you have VRAM headroom

device = "cuda" if torch.cuda.is_available() else "cpu"

compute_dtype = torch.float16  # avoid bf16 issues on Colab

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading reference (base) model in 4-bit...")
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},              # force to GPU for this pass
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
)
ref_model.eval()

def build_user_prompt(prompt_text: str) -> str:
    return (
        "You are answering a questionnaire as an individual person. "
        "Respond naturally and thoughtfully, as someone would in real life. "
        "Do not mention being an AI or assistant. "
        "Keep the answer short, under 3 sentences. "
        "Give a sincere, human-like answer.\n\n"
        "Situation:\n"
        f"{prompt_text.strip()}\n\n"
        "Answer:"
    )


def format_prompt_text(tokenizer, prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": build_user_prompt(prompt_text)}],
        tokenize=False,
        add_generation_prompt=True,
    )


def format_dataset_example(ex):
    ex["prompt"] = format_prompt_text(tokenizer, ex["prompt"])
    return ex

@torch.no_grad()
def seq_logprob_for_completion(prompt_text: str, completion_text: str) -> float:
    """
    Returns log p(completion | prompt) under ref_model, summed over completion tokens.
    Prompt tokens are masked out.
    """
    prompt = format_prompt_text(tokenizer, prompt_text)

    # Tokenize prompt with truncation (keep end tends to preserve immediate context)
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    if len(prompt_ids) > MAX_PROMPT_TOKENS:
        prompt_ids = prompt_ids[-MAX_PROMPT_TOKENS:]

    # Tokenize completion
    comp_ids = tokenizer(completion_text, add_special_tokens=False).input_ids
    if len(comp_ids) > MAX_COMPLETION_TOKENS:
        comp_ids = comp_ids[:MAX_COMPLETION_TOKENS]

    input_ids = prompt_ids + comp_ids
    if len(input_ids) < 2:
        return float("-inf")

    # Build labels: ignore prompt tokens, score completion tokens
    labels = ([-100] * len(prompt_ids)) + comp_ids

    # Convert to tensors
    input_ids_t = torch.tensor([input_ids], device=ref_model.device)
    labels_t    = torch.tensor([labels], device=ref_model.device)

    outputs = ref_model(input_ids=input_ids_t)
    logits = outputs.logits  # [1, seq, vocab]

    log_probs = torch.log_softmax(logits, dim=-1)

    # Shift for next-token prediction
    log_probs = log_probs[:, :-1, :]
    labels_shifted = labels_t[:, 1:]

    mask = labels_shifted.ne(-100)
    # gather logp for gold tokens
    gold = labels_shifted.clone()
    gold[~mask] = 0
    token_logps = log_probs.gather(-1, gold.unsqueeze(-1)).squeeze(-1)
    token_logps = token_logps * mask

    return float(token_logps.sum().detach().cpu().to(torch.float32))

# Read all data
with open(IN_FILE, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(data)} examples from {IN_FILE}")


Loading tokenizer...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading reference (base) model in 4-bit...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded 280 examples from /content/drive/MyDrive/DPO/D_syn_USA_train.jsonl


In [ ]:
# Compute and write
with open(OUT_FILE, "w", encoding="utf-8") as out:
    for i, ex in enumerate(data, 1):
        p = ex["prompt"]
        chosen = ex["chosen"]
        rejected = ex["rejected"]

        ref_chosen_logps = seq_logprob_for_completion(p, chosen)
        ref_rejected_logps = seq_logprob_for_completion(p, rejected)

        ex["ref_chosen_logps"] = ref_chosen_logps
        ex["ref_rejected_logps"] = ref_rejected_logps

        out.write(json.dumps(ex, ensure_ascii=False) + "\n")

        if i % 25 == 0:
            print(f"Precomputed {i}/{len(data)}")

print("Done. Wrote:", OUT_FILE)

# Free VRAM
del ref_model
torch.cuda.empty_cache()

Precomputed 25/280
Precomputed 50/280
Precomputed 75/280
Precomputed 100/280
Precomputed 125/280
Precomputed 150/280
Precomputed 175/280
Precomputed 200/280
Precomputed 225/280
Precomputed 250/280
Precomputed 275/280
Done. Wrote: /content/drive/MyDrive/DPO/D_syn_USA_train_with_ref.jsonl


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

# Quick VRAM check
free, total = torch.cuda.mem_get_info()
print(f"Free VRAM: {free/1024**3:.2f} GiB / Total: {total/1024**3:.2f} GiB")

compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},          # FORCE everything onto GPU 0
    #torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,   # force
)

print("Loaded OK")



Free VRAM: 11.01 GiB / Total: 14.56 GiB


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded OK


In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

DATA_FILE  = f"/content/drive/MyDrive/DPO/D_syn_{DATA_COUNTRY_CODE}_train_with_ref.jsonl"
OUTPUT_DIR = f"/content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_{ADAPTER_TAG}"


compute_dtype = torch.float16  # force fp16 to avoid bf16 AMP errors

# Training setup
model.config.use_cache = False

# Manual fallback if needed:
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False

##############

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = load_dataset("json", data_files={"train": DATA_FILE})
train_dataset = ds["train"]

train_dataset = train_dataset.map(format_dataset_example)

##################

peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_cfg)

for name, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float16)

# Also cast any floating buffers (rarely needed, but safe)
for name, b in model.named_buffers():
    if b.is_floating_point():
        try:
            model._buffers[name] = b.to(torch.float16)
        except Exception:
            pass

# Sanity check: should now be float16
trainable = next(p for p in model.parameters() if p.requires_grad)
print("Trainable param dtype:", trainable.dtype)

model.print_trainable_parameters()

dpo_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=0.1,
    max_length=768,             # lower this if VRAM is tight
    truncation_mode="keep_end",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to="none",

    fp16=True,
    bf16=False,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,

)

def force_trainable_fp32(m):
    for n, p in m.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.to(torch.float32)

force_trainable_fp32(trainer.model)

trainable = next(p for p in trainer.model.parameters() if p.requires_grad)
print("Trainable dtype after trainer init:", trainable.dtype)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/280 [00:00<?, ? examples/s]

/tmp/ipykernel_9227/1393693144.py:78: FutureWarning: The `'keep_end'` truncation mode is deprecated and will be removed in v2.0.0. Use `truncation_mode='keep_start'` (the default) instead.
  dpo_args = DPOConfig(
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainable param dtype: torch.float16
trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


Adding EOS to train dataset:   0%|          | 0/280 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/280 [00:00<?, ? examples/s]

Trainable dtype after trainer init: torch.float32


In [ ]:
trainable = next(p for p in model.parameters() if p.requires_grad)
print("Trainable param dtype:", trainable.dtype)
print("Model param dtype:", next(p for p in model.parameters() if p.requires_grad).dtype)
# After trainer is created:
print("Accelerate mixed_precision:", trainer.accelerator.mixed_precision)

bf16_names = [n for n,p in model.named_parameters() if p.requires_grad and p.dtype == torch.bfloat16]
print("BF16 trainable params:", bf16_names[:20], "count:", len(bf16_names))

Trainable param dtype: torch.float32
Model param dtype: torch.float32
Accelerate mixed_precision: fp16
BF16 trainable params: [] count: 0


In [ ]:
trainer.train()

# Paste output diresctory here
OUTPUT_DIR = f"/content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_{ADAPTER_TAG}"
# Save adapter-only
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapter-only model to:", OUTPUT_DIR)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
10,0.455078


Saved adapter-only model to: /content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_US


## Loading and testing.

Reload runtime to save memory.

In [ ]:
!pip -q install -U "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.0" \
                 "trl>=0.11.0" "peft>=0.11.1" "bitsandbytes>=0.46.1" "safetensors>=0.4.3"

In [ ]:
#!pip install -U "huggingface_hub[cli]"
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def build_user_prompt(prompt_text: str) -> str:
    return (
        "You are answering a questionnaire as an individual person. "
        "Respond naturally and thoughtfully, as someone would in real life. "
        "Do not mention being an AI or assistant. "
        "Keep the answer short, under 3 sentences. "
        "Give a sincere, human-like answer.\n\n"
        "Situation:\n"
        f"{prompt_text.strip()}\n\n"
        "Answer:"
    )


def format_prompt_text(tokenizer, prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": build_user_prompt(prompt_text)}],
        tokenize=False,
        add_generation_prompt=True,
    )

In [ ]:
import math
import torch
import pandas as pd
from tqdm import tqdm

BETA = 0.1


def get_model_device(model):
    """
    Returns the device where the model is located.
    Works for regular models and PEFT/QLoRA models.
    """
    return next(model.parameters()).device


@torch.no_grad()
def sequence_logprob(
    model,
    tokenizer,
    prompt_text,
    completion_text,
    max_prompt_tokens=512,
    max_completion_tokens=256,
):
    """
    Computes log p(completion | prompt) for a single prompt-completion pair.

    Important:
    This should use the same prompt formatting as DPO training.
    The log probability is summed over completion tokens only.
    """

    model.eval()
    device = get_model_device(model)

    # Use the same chat-template format used during DPO training.

    formatted_prompt = format_prompt_text(tokenizer, prompt_text)

    prompt_ids = tokenizer(
        formatted_prompt,
        add_special_tokens=False,
    ).input_ids

    completion_ids = tokenizer(
        completion_text,
        add_special_tokens=False,
    ).input_ids

    # Truncate if needed.
    if len(prompt_ids) > max_prompt_tokens:
        prompt_ids = prompt_ids[-max_prompt_tokens:]

    if len(completion_ids) > max_completion_tokens:
        completion_ids = completion_ids[:max_completion_tokens]

    input_ids = prompt_ids + completion_ids

    # Mask prompt tokens so only completion tokens count in the loss/logprob.
    labels = [-100] * len(prompt_ids) + completion_ids

    input_ids = torch.tensor([input_ids], device=device)
    labels = torch.tensor([labels], device=device)

    outputs = model(input_ids=input_ids)
    logits = outputs.logits

    # Shift for next-token prediction.
    shifted_logits = logits[:, :-1, :]
    shifted_labels = labels[:, 1:]

    log_probs = torch.log_softmax(shifted_logits, dim=-1)

    mask = shifted_labels.ne(-100)

    safe_labels = shifted_labels.clone()
    safe_labels[~mask] = 0

    token_log_probs = log_probs.gather(
        dim=-1,
        index=safe_labels.unsqueeze(-1),
    ).squeeze(-1)

    completion_logprob = (token_log_probs * mask).sum()

    return float(completion_logprob.detach().cpu())

def dpo_implied_reward_delta(
    adapter_model,
    tokenizer,
    prompt,
    chosen,
    rejected,
    beta=BETA,
):
    """
    Computes the DPO implied reward difference using the same PEFT model:

        ref logprobs: adapter disabled
        adapter logprobs: adapter enabled

    This avoids accidentally comparing the adapter to itself.
    """

    # Reference model: same base, adapter disabled.
    with adapter_model.disable_adapter():
        ref_chosen_logp = sequence_logprob(
            adapter_model,
            tokenizer,
            prompt,
            chosen,
        )

        ref_rejected_logp = sequence_logprob(
            adapter_model,
            tokenizer,
            prompt,
            rejected,
        )

    # Adapter model: adapter enabled.
    adapter_chosen_logp = sequence_logprob(
        adapter_model,
        tokenizer,
        prompt,
        chosen,
    )

    adapter_rejected_logp = sequence_logprob(
        adapter_model,
        tokenizer,
        prompt,
        rejected,
    )

    ref_margin = ref_chosen_logp - ref_rejected_logp
    adapter_margin = adapter_chosen_logp - adapter_rejected_logp

    reward_delta = beta * (adapter_margin - ref_margin)

    dpo_pref_prob = 1.0 / (1.0 + math.exp(-reward_delta))

    return {
        "ref_chosen_logp": ref_chosen_logp,
        "ref_rejected_logp": ref_rejected_logp,
        "adapter_chosen_logp": adapter_chosen_logp,
        "adapter_rejected_logp": adapter_rejected_logp,
        "ref_margin": ref_margin,
        "adapter_margin": adapter_margin,
        "dpo_reward_delta": reward_delta,
        "dpo_pref_prob": dpo_pref_prob,
        "dpo_prefers_chosen": reward_delta > 0,
    }

@torch.no_grad()
def generate_model_answer(
    model,
    tokenizer,
    prompt_text,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
):
    """
    Generates a free-form answer from the model for a prompt.
    Uses the same prompt format as reward recovery and training.
    """

    model.eval()
    device = get_model_device(model)

    formatted_prompt = format_prompt_text(tokenizer, prompt_text)

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Keep only newly generated tokens, not the prompt.
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return generated_text

def evaluate_adapter_reward_recovery(
    adapter_model,
    tokenizer,
    eval_file,
    model_name,
    eval_country,
    beta=BETA,
    max_examples=None,
    generate_answers=True,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
):
    """
    Runs DPO implied reward recovery on a held-out eval JSONL file.

    Also optionally generates a free-form answer from the adapter
    for each prompt and stores it in the result dataframe.
    """

    rows = load_jsonl(eval_file)

    if max_examples is not None:
        rows = rows[:max_examples]

    results = []

    for ex in tqdm(rows, desc=f"Reward recovery: {model_name} on {eval_country}"):
        out = dpo_implied_reward_delta(
            adapter_model=adapter_model,
            tokenizer=tokenizer,
            prompt=ex["prompt"],
            chosen=ex["chosen"],
            rejected=ex["rejected"],
            beta=beta,
        )

        if generate_answers:
            generated_answer = generate_model_answer(
                model=adapter_model,
                tokenizer=tokenizer,
                prompt_text=ex["prompt"],
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )
        else:
            generated_answer = None

        results.append({
            "model": model_name,
            "eval_country": eval_country,
            "country": ex.get("country"),
            "item_id": ex.get("item_id"),
            #"dimension": ex.get("dimension"),
            "gps_dimension": ex.get("gps_dimension"),
            "prompt": ex["prompt"],
            "chosen": ex["chosen"],
            "rejected": ex["rejected"],
            "generated_answer": generated_answer,
            **out,
        })

    return pd.DataFrame(results)



In [ ]:
import gc
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch
import pandas as pd

# Paste directories here as needed
BASE_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
US_ADAPTER_DIR  = "/content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_US"
MEX_ADAPTER_DIR = "/content/drive/MyDrive/DPO/dpo_qlora_adapter_llama3_MEX"

US_EVAL_FILE  = "/content/drive/MyDrive/DPO/D_syn_USA_eval.jsonl"
MEX_EVAL_FILE = "/content/drive/MyDrive/DPO/D_syn_MEX_eval.jsonl"

RESULTS_DIR = Path("/content/drive/MyDrive/DPO/eval_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BETA = 0.1


def load_base_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )

    base_model.eval()

    return base_model, tokenizer


def load_adapter_model(adapter_dir):
    base_model, tokenizer = load_base_model_and_tokenizer()

    adapter_model = PeftModel.from_pretrained(
        base_model,
        adapter_dir,
    )

    adapter_model.eval()

    return adapter_model, tokenizer


def cleanup_models(*models):
    for model in models:
        del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
import json
import random
from pathlib import Path


DATA_DIR = Path("/content/drive/MyDrive/DPO")

US_RAW_FILE  = DATA_DIR / "D_syn_USA_350.jsonl"
MEX_RAW_FILE = DATA_DIR / "D_syn_MEX_350.jsonl"

US_TRAIN_FILE  = DATA_DIR / "D_syn_USA_train.jsonl"
US_EVAL_FILE   = DATA_DIR / "D_syn_USA_eval.jsonl"

MEX_TRAIN_FILE = DATA_DIR / "D_syn_MEX_train.jsonl"
MEX_EVAL_FILE  = DATA_DIR / "D_syn_MEX_eval.jsonl"

TRAIN_FRAC = 0.80
SEED = 42


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def split_country_file(raw_path, train_path, eval_path, country, train_frac=0.80, seed=42):
    rows = load_jsonl(raw_path)

    # Add useful metadata for later validation.
    for i, row in enumerate(rows):
        row.setdefault("country", country)
        row.setdefault("item_id", f"{country}_{i:04d}")

    rng = random.Random(seed)
    rng.shuffle(rows)

    n_train = int(len(rows) * train_frac)

    train_rows = rows[:n_train]
    eval_rows = rows[n_train:]

    write_jsonl(train_rows, train_path)
    write_jsonl(eval_rows, eval_path)

    print(f"{country}: {len(rows)} total")
    print(f"  train: {len(train_rows)} -> {train_path}")
    print(f"  eval:  {len(eval_rows)} -> {eval_path}")

    return train_rows, eval_rows


In [ ]:
# ============================
# Evaluate US adapter
# ============================

us_model, tokenizer = load_adapter_model(US_ADAPTER_DIR)

df_us_adapter_on_us = evaluate_adapter_reward_recovery(
    adapter_model=us_model,
    tokenizer=tokenizer,
    eval_file=US_EVAL_FILE,
    model_name="US_adapter",
    eval_country="US",
    beta=BETA,
    generate_answers=True,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
)
df_us_adapter_on_mex = evaluate_adapter_reward_recovery(
    adapter_model=us_model,
    tokenizer=tokenizer,
    eval_file=MEX_EVAL_FILE,
    model_name="US_adapter",
    eval_country="Mexico",
    beta=BETA,
    generate_answers=True,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
)

df_us_adapter = pd.concat(
    [df_us_adapter_on_us, df_us_adapter_on_mex],
    ignore_index=True,
)

df_us_adapter.to_csv(
    RESULTS_DIR / "reward_recovery_US_adapter1.csv",
    index=False,
)

df_us_adapter.head()

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


Reward recovery: US_adapter on US:   0%|          | 0/70 [00:00<?, ?it/s][transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.

Reward recovery: US_adapter on US: 100%|██████████| 70/70 [16:04<00:00, 13.77s/it]

Reward recovery: US_adapter on Mexico: 100%|██████████| 70/70 [16:14<00:00, 13.93s/it]


,model,eval_country,country,item_id,gps_dimension,prompt,chosen,rejected,generated_answer,ref_chosen_logp,ref_rejected_logp,adapter_chosen_logp,adapter_rejected_logp,ref_margin,adapter_margin,dpo_reward_delta,dpo_pref_prob,dpo_prefers_chosen
0,US_adapter,US,USA,US_0136,altruism,Your employer offers a donation-matching progr...,I’ll go with the less familiar organization—ev...,I’m designating the cancer research foundation...,I'm going to designate my donation to the smal...,-185.0,-176.0,-171.0,-172.0,-9.0,1.0,1.00,0.731059,True
1,US_adapter,US,USA,US_0107,altruism,A rebate check arrives in the mail for a produ...,I’d go ahead and donate the rebate through the...,I’d probably keep the rebate for myself—it’s n...,I'd definitely take a moment to make a donatio...,-150.0,-177.0,-129.0,-189.0,27.0,60.0,3.30,0.964429,True
2,US_adapter,US,USA,US_0181,risktaking,You agreed to mentor a struggling student thro...,I’ll double down—this kid just needs the right...,I’m scaling back the intensity to avoid burnou...,I absolutely remain committed to mentoring the...,-190.0,-175.0,-181.0,-174.0,-15.0,-7.0,0.80,0.689974,True
3,US_adapter,US,USA,US_0189,patience,You baked a batch of cookies and the recipe sa...,I’ll set a timer for the full thirty minutes a...,I’ll grab one now—they smell too good to resis...,I think I'll just wait the remaining twenty mi...,-114.5,-122.5,-112.0,-112.5,8.0,0.5,-0.75,0.320821,False
4,US_adapter,US,USA,US_0083,trust,Your online purchase arrives three days later ...,I’ll give the seller the benefit of the doubt—...,This feels like a red flag—if they can’t meet ...,I'll give the seller the benefit of the doubt ...,-145.0,-179.0,-122.0,-197.0,34.0,75.0,4.10,0.983698,True


In [ ]:
cleanup_models(us_model)

In [ ]:
# ============================
# Evaluate Mexico adapter
# ============================

mex_model, tokenizer = load_adapter_model(MEX_ADAPTER_DIR)

df_mex_adapter_on_us = evaluate_adapter_reward_recovery(
    adapter_model=mex_model,
    tokenizer=tokenizer,
    eval_file=US_EVAL_FILE,
    model_name="Mexico_adapter",
    eval_country="US",
    beta=BETA,
    generate_answers=True,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
)

df_mex_adapter_on_mex = evaluate_adapter_reward_recovery(
    adapter_model=mex_model,
    tokenizer=tokenizer,
    eval_file=MEX_EVAL_FILE,
    model_name="Mexico_adapter",
    eval_country="Mexico",
    beta=BETA,
    generate_answers=True,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9,
)

df_mex_adapter = pd.concat(
    [df_mex_adapter_on_us, df_mex_adapter_on_mex],
    ignore_index=True,
)

df_mex_adapter.to_csv(
    RESULTS_DIR / "reward_recovery_Mexico_adapter1.csv",
    index=False,
)

df_mex_adapter.head()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Reward recovery: Mexico_adapter on Mexico: 100%|██████████| 70/70 [14:15<00:00, 12.22s/it]


,model,eval_country,country,item_id,gps_dimension,prompt,chosen,rejected,generated_answer,ref_chosen_logp,ref_rejected_logp,adapter_chosen_logp,adapter_rejected_logp,ref_margin,adapter_margin,dpo_reward_delta,dpo_pref_prob,dpo_prefers_chosen
0,Mexico_adapter,US,USA,US_0136,altruism,Your employer offers a donation-matching progr...,I’ll go with the less familiar organization—ev...,I’m designating the cancer research foundation...,I think I'd choose the local cancer research f...,-185.0,-176.0,-208.0,-176.0,-9.0,-32.0,-2.30,0.091123,False
1,Mexico_adapter,US,USA,US_0107,altruism,A rebate check arrives in the mail for a produ...,I’d go ahead and donate the rebate through the...,I’d probably keep the rebate for myself—it’s n...,"Honestly, I might not use it for the charity d...",-150.0,-177.0,-156.0,-187.0,27.0,31.0,0.40,0.598688,True
2,Mexico_adapter,US,USA,US_0181,risktaking,You agreed to mentor a struggling student thro...,I’ll double down—this kid just needs the right...,I’m scaling back the intensity to avoid burnou...,"Honestly, I'm starting to feel a bit overwhelm...",-190.0,-175.0,-208.0,-205.0,-15.0,-3.0,1.20,0.768525,True
3,Mexico_adapter,US,USA,US_0189,patience,You baked a batch of cookies and the recipe sa...,I’ll set a timer for the full thirty minutes a...,I’ll grab one now—they smell too good to resis...,"I don't know, I think I'll just let them cool ...",-114.5,-122.5,-120.0,-115.5,8.0,-4.5,-1.25,0.222700,False
4,Mexico_adapter,US,USA,US_0083,trust,Your online purchase arrives three days later ...,I’ll give the seller the benefit of the doubt—...,This feels like a red flag—if they can’t meet ...,I think I'll just give them the benefit of the...,-145.0,-179.0,-149.0,-217.0,34.0,68.0,3.40,0.967705,True


In [ ]:
cleanup_models(mex_model)

In [ ]:
# ============================
# Combine adapter reward-recovery results
# ============================

adapter_results = pd.concat(
    [df_us_adapter, df_mex_adapter],
    ignore_index=True,
)

adapter_results.to_csv(
    RESULTS_DIR / "reward_recovery_adapters_combined1.csv",
    index=False,
)

adapter_summary = (
    adapter_results
    .groupby(["model", "eval_country"])
    .agg(
        n=("dpo_reward_delta", "size"),
        mean_reward_delta=("dpo_reward_delta", "mean"),
        median_reward_delta=("dpo_reward_delta", "median"),
        preference_accuracy=("dpo_prefers_chosen", "mean"),
        mean_dpo_pref_prob=("dpo_pref_prob", "mean"),
        mean_ref_margin=("ref_margin", "mean"),
        mean_adapter_margin=("adapter_margin", "mean"),
    )
    .reset_index()
)

adapter_summary.to_csv(
    RESULTS_DIR / "reward_recovery_adapter_summary1.csv",
    index=False,
)

adapter_summary

,model,eval_country,n,mean_reward_delta,median_reward_delta,preference_accuracy,mean_dpo_pref_prob,mean_ref_margin,mean_adapter_margin
0,Mexico_adapter,Mexico,70,3.102857,2.975,0.985714,0.908947,-8.721429,22.307143
1,Mexico_adapter,US,70,-1.028571,-1.175,0.214286,0.315670,-8.707143,-18.992857
2,US_adapter,Mexico,70,-1.226429,-1.375,0.271429,0.301702,-8.721429,-20.985714
3,US_adapter,US,70,3.239286,3.325,0.971429,0.907736,-8.707143,23.685714


In [ ]:
# ============================
# Specialization scores
# ============================

summary_wide = adapter_summary.pivot(
    index="model",
    columns="eval_country",
    values="mean_reward_delta",
)

summary_wide["own_minus_other"] = None

if "US_adapter" in summary_wide.index:
    summary_wide.loc["US_adapter", "own_minus_other"] = (
        summary_wide.loc["US_adapter", "US"]
        - summary_wide.loc["US_adapter", "Mexico"]
    )

if "Mexico_adapter" in summary_wide.index:
    summary_wide.loc["Mexico_adapter", "own_minus_other"] = (
        summary_wide.loc["Mexico_adapter", "Mexico"]
        - summary_wide.loc["Mexico_adapter", "US"]
    )

summary_wide

eval_country,Mexico,US,own_minus_other
model,,,
Mexico_adapter,3.102857,-1.028571,4.131429
US_adapter,-1.226429,3.239286,4.465714


In [ ]:
dimension_summary = (
    adapter_results
    .dropna(subset=["gps_dimension"])
    .groupby(["model", "eval_country", "gps_dimension"])
    .agg(
        n=("dpo_reward_delta", "size"),
        mean_reward_delta=("dpo_reward_delta", "mean"),
        preference_accuracy=("dpo_prefers_chosen", "mean"),
        mean_dpo_pref_prob=("dpo_pref_prob", "mean"),
    )
    .reset_index()
)

dimension_summary.to_csv(
    RESULTS_DIR / "reward_recovery_dimension_summary1.csv",
    index=False,
)

dimension_summary

,model,eval_country,gps_dimension,n,mean_reward_delta,preference_accuracy,mean_dpo_pref_prob
0,Mexico_adapter,Mexico,altruism,12,4.025000,1.000000,0.954556
1,Mexico_adapter,Mexico,negrecip,12,3.791667,1.000000,0.946196
2,Mexico_adapter,Mexico,patience,8,1.962500,0.875000,0.807364
3,Mexico_adapter,Mexico,posrecip,17,2.914706,1.000000,0.896763
4,Mexico_adapter,Mexico,risktaking,10,2.550000,1.000000,0.895036
5,Mexico_adapter,Mexico,trust,11,2.968182,1.000000,0.923909
6,Mexico_adapter,US,altruism,26,-1.628846,0.076923,0.207742
7,Mexico_adapter,US,patience,12,-1.350000,0.166667,0.268485
8,Mexico_adapter,US,posrecip,1,-2.900000,0.000000,0.052154
9,Mexico_adapter,US,risktaking,19,-0.323684,0.473684,0.454395
